In [3]:
import os
import pandas as pd
import numpy as np

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
dataset_path = "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1"

In [6]:
driver = "D1"

trip = "20151110175712-16km-D1-NORMAL1-SECONDARY"

trip_path = os.path.join(
    dataset_path,
    driver,
    trip
)

In [7]:
acc_path = os.path.join(
    trip_path,
    "RAW_ACCELEROMETERS.txt"
)

gps_path = os.path.join(
    trip_path,
    "RAW_GPS.txt"
)

In [8]:
print(acc_path)
print(gps_path)

/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1/D1/20151110175712-16km-D1-NORMAL1-SECONDARY/RAW_ACCELEROMETERS.txt
/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1/D1/20151110175712-16km-D1-NORMAL1-SECONDARY/RAW_GPS.txt


In [9]:
acc_df = load_accelerometer(acc_path)

acc_df.head()

,timestamp,active,acc_x,acc_y,acc_z,acc_x_kf,acc_y_kf,acc_z_kf,roll,pitch,yaw
0,6.94,1,0.017,-0.011,0.018,-0.005,0.008,0.018,-1.523,0.015,0.012
1,7.03,1,0.046,0.007,0.019,0.016,-0.002,0.018,-1.522,0.012,0.012
2,7.14,1,0.052,-0.016,0.027,0.037,-0.005,0.018,-1.520,0.014,0.011
3,7.24,1,0.015,-0.016,0.026,0.038,-0.009,0.024,-1.523,0.014,0.011
4,7.34,1,-0.014,-0.017,0.040,0.012,-0.016,0.032,-1.525,0.012,0.011


In [10]:
def load_gps(gps_path):

    gps_columns = [

        "timestamp",
        "speed",
        "latitude",
        "longitude",
        "altitude",

        "gps_quality",
        "satellites",

        "heading",

        "extra_1",
        "extra_2",
        "extra_3",
        "extra_4"

    ]

    gps_df = pd.read_csv(
        gps_path,
        sep=r"\s+",
        header=None,
        names=gps_columns
    )

    return gps_df

In [11]:
gps_df = load_gps(gps_path)

gps_df.head()

,timestamp,speed,latitude,longitude,altitude,gps_quality,satellites,heading,extra_1,extra_2,extra_3,extra_4
0,7.85,65.2,40.512787,-3.404477,612.7,4,5,331.9,0.000,0,0,0
1,8.83,64.5,40.512924,-3.404577,612.5,4,5,331.9,0.000,0,0,0
2,9.82,63.6,40.513065,-3.404680,612.9,4,5,330.8,1.055,0,0,0
3,10.80,62.2,40.513210,-3.404772,613.3,4,5,330.8,1.055,0,0,0
4,11.80,60.9,40.513348,-3.404868,613.5,3,5,330.1,0.703,0,0,0


In [12]:
def synchronize_sensors(acc_df, gps_df):

    master_df = pd.merge_asof(

        acc_df.sort_values("timestamp"),

        gps_df.sort_values("timestamp"),

        on="timestamp",

        direction="nearest"

    )

    return master_df

In [13]:
master_df = synchronize_sensors(
    acc_df,
    gps_df
)

master_df.head()

,timestamp,active,acc_x,acc_y,acc_z,acc_x_kf,acc_y_kf,acc_z_kf,roll,pitch,...,latitude,longitude,altitude,gps_quality,satellites,heading,extra_1,extra_2,extra_3,extra_4
0,6.94,1,0.017,-0.011,0.018,-0.005,0.008,0.018,-1.523,0.015,...,40.512787,-3.404477,612.7,4,5,331.9,0.0,0,0,0
1,7.03,1,0.046,0.007,0.019,0.016,-0.002,0.018,-1.522,0.012,...,40.512787,-3.404477,612.7,4,5,331.9,0.0,0,0,0
2,7.14,1,0.052,-0.016,0.027,0.037,-0.005,0.018,-1.520,0.014,...,40.512787,-3.404477,612.7,4,5,331.9,0.0,0,0,0
3,7.24,1,0.015,-0.016,0.026,0.038,-0.009,0.024,-1.523,0.014,...,40.512787,-3.404477,612.7,4,5,331.9,0.0,0,0,0
4,7.34,1,-0.014,-0.017,0.040,0.012,-0.016,0.032,-1.525,0.012,...,40.512787,-3.404477,612.7,4,5,331.9,0.0,0,0,0


In [14]:
print(master_df.shape)

master_df.info()

(6170, 22)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6170 entries, 0 to 6169
Data columns (total 22 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   timestamp    6170 non-null   float64
 1   active       6170 non-null   int64  
 2   acc_x        6170 non-null   float64
 3   acc_y        6170 non-null   float64
 4   acc_z        6170 non-null   float64
 5   acc_x_kf     6170 non-null   float64
 6   acc_y_kf     6170 non-null   float64
 7   acc_z_kf     6170 non-null   float64
 8   roll         6170 non-null   float64
 9   pitch        6170 non-null   float64
 10  yaw          6170 non-null   float64
 11  speed        6170 non-null   float64
 12  latitude     6170 non-null   float64
 13  longitude    6170 non-null   float64
 14  altitude     6170 non-null   float64
 15  gps_quality  6170 non-null   int64  
 16  satellites   6170 non-null   int64  
 17  heading      6170 non-null   float64
 18  extra_1      6170 non-null   float64


In [18]:
def engineer_features(master_df):

    master_df = master_df.copy()

    # -------------------------------------------------
    # Acceleration Features
    # -------------------------------------------------

    master_df["acc_resultant"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2 +
        master_df["acc_z"]**2
    )

    master_df["acc_horizontal"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2
    )

    master_df["acc_vertical"] = master_df["acc_z"]

    # -------------------------------------------------
    # Delta Features
    # -------------------------------------------------

    master_df["speed_delta"] = master_df["speed"].diff().fillna(0)

    master_df["heading_delta"] = master_df["heading"].diff().fillna(0)

    master_df["roll_delta"] = master_df["roll"].diff().fillna(0)

    master_df["pitch_delta"] = master_df["pitch"].diff().fillna(0)

    master_df["yaw_delta"] = master_df["yaw"].diff().fillna(0)

    return master_df

In [22]:
feature_df = engineer_features(master_df)

feature_df[
    [
        "speed_delta",
        "heading_delta",
        "roll_delta",
        "pitch_delta",
        "yaw_delta"
    ]
].head()

,speed_delta,heading_delta,roll_delta,pitch_delta,yaw_delta
0,0.0,0.0,0.000,0.000,0.000
1,0.0,0.0,0.001,-0.003,0.000
2,0.0,0.0,0.002,0.002,-0.001
3,0.0,0.0,-0.003,0.000,0.000
4,0.0,0.0,-0.002,-0.002,0.000


In [23]:
delta_features = [

    "speed_delta",

    "heading_delta",

    "roll_delta",

    "pitch_delta",

    "yaw_delta"

]

feature_df[delta_features].describe().T

,count,mean,std,min,25%,50%,75%,max
speed_delta,6170.0,0.003663,0.413470,-6.000,0.000,0.0,0.000,9.700
heading_delta,6170.0,-0.008720,0.526080,-6.400,0.000,0.0,0.000,6.300
roll_delta,6170.0,-0.000001,0.001968,-0.011,-0.001,0.0,0.001,0.018
pitch_delta,6170.0,-0.000004,0.003188,-0.025,-0.002,0.0,0.001,0.039
yaw_delta,6170.0,-0.000209,0.002354,-0.018,-0.001,0.0,0.001,0.009


In [24]:
def extract_statistics(signal):

    features = {}

    features["mean"] = signal.mean()

    features["std"] = signal.std()

    features["min"] = signal.min()

    features["max"] = signal.max()

    features["median"] = signal.median()

    features["rms"] = np.sqrt(
        np.mean(signal**2)
    )

    return features

In [25]:
extract_statistics(
    feature_df["speed"]
)

{'mean': np.float64(96.21596434359806),
 'std': 9.59643596976475,
 'min': 60.9,
 'max': 116.8,
 'median': 95.4,
 'rms': np.float64(96.6932699425767)}

In [26]:
extract_statistics(
    feature_df["acc_resultant"]
)

{'mean': np.float64(0.0473157013168258),
 'std': 0.028211281693656578,
 'min': 0.002449489742783178,
 'max': 0.2522558225294314,
 'median': 0.041719299659639275,
 'rms': np.float64(0.055086504831825284)}

In [27]:
def extract_window_features(window, feature_list):

    window_stats = {}

    for feature in feature_list:

        stats = extract_statistics(window[feature])

        for stat_name, stat_value in stats.items():

            column_name = f"{feature}_{stat_name}"

            window_stats[column_name] = stat_value

    return window_stats

In [28]:
window_features = [

    "acc_resultant",

    "acc_horizontal",

    "speed",

    "speed_delta",

    "roll",

    "pitch",

    "yaw"

]

In [29]:
WINDOW_SIZE = 30

first_window = feature_df.iloc[:WINDOW_SIZE]

In [30]:
window_dict = extract_window_features(
    first_window,
    window_features
)

len(window_dict)

42

In [45]:
def create_sliding_windows(
        feature_df,
        feature_list,
        window_size
):

    all_window_features = []

    for start in range(
        0,
        len(feature_df) - window_size + 1
    ):

        end = start + window_size

        window = feature_df.iloc[start:end]

        window_stats = extract_window_features(
            window,
            feature_list
        )

        all_window_features.append(window_stats)

    return pd.DataFrame(all_window_features)

In [37]:
window_dataset = create_sliding_windows(
    feature_df,
    window_features,
    WINDOW_SIZE
)

len(window_dataset)

6141

In [38]:
window_dataset[0]

{'acc_resultant_mean': np.float64(0.04155660035870177),
 'acc_resultant_std': 0.015206020811659945,
 'acc_resultant_min': 0.019339079605813717,
 'acc_resultant_max': 0.07522632517942107,
 'acc_resultant_median': 0.037946883553436825,
 'acc_resultant_rms': np.float64(0.044164087975035404),
 'acc_horizontal_mean': np.float64(0.036258717121429515),
 'acc_horizontal_std': 0.0156779693675933,
 'acc_horizontal_min': 0.012206555615733704,
 'acc_horizontal_max': 0.07516648189186455,
 'acc_horizontal_median': 0.03312853668746009,
 'acc_horizontal_rms': np.float64(0.039399238571322666),
 'speed_mean': np.float64(64.66999999999999),
 'speed_std': 0.6254102102096539,
 'speed_min': 63.6,
 'speed_max': 65.2,
 'speed_median': 64.85,
 'speed_rms': np.float64(64.67292323685393),
 'speed_delta_mean': np.float64(-0.05333333333333338),
 'speed_delta_std': 0.20465839213495354,
 'speed_delta_min': -0.8999999999999986,
 'speed_delta_max': 0.0,
 'speed_delta_median': 0.0,
 'speed_delta_rms': np.float64(0.2081

In [39]:
len(window_dataset[0])

42

In [40]:
window_dataset = pd.DataFrame(window_dataset)

In [41]:
window_dataset.head()

,acc_resultant_mean,acc_resultant_std,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_horizontal_mean,acc_horizontal_std,acc_horizontal_min,acc_horizontal_max,...,pitch_min,pitch_max,pitch_median,pitch_rms,yaw_mean,yaw_std,yaw_min,yaw_max,yaw_median,yaw_rms
0,0.041557,0.015206,0.019339,0.075226,0.037947,0.044164,0.036259,0.015678,0.012207,0.075166,...,0.012,0.032,0.0220,0.021821,0.019100,0.005182,0.011,0.025,0.0215,0.019768
1,0.041732,0.015063,0.019339,0.075226,0.037947,0.044282,0.036556,0.015447,0.012207,0.075166,...,0.012,0.032,0.0220,0.022052,0.019533,0.005111,0.011,0.025,0.0220,0.020169
2,0.041103,0.015089,0.019339,0.075226,0.036665,0.043698,0.035783,0.015511,0.012207,0.075166,...,0.012,0.032,0.0225,0.022413,0.020033,0.005082,0.011,0.027,0.0220,0.020647
3,0.039805,0.015016,0.019339,0.075226,0.034974,0.042455,0.034164,0.016026,0.005831,0.075166,...,0.012,0.032,0.0230,0.022767,0.020667,0.005101,0.011,0.030,0.0220,0.021267
4,0.040324,0.015078,0.019339,0.075226,0.036665,0.042963,0.035077,0.016086,0.005831,0.075166,...,0.012,0.032,0.0230,0.023116,0.021333,0.005101,0.011,0.031,0.0225,0.021915


In [42]:
window_dataset.shape

(6141, 42)

In [43]:
window_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6141 entries, 0 to 6140
Data columns (total 42 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   acc_resultant_mean     6141 non-null   float64
 1   acc_resultant_std      6141 non-null   float64
 2   acc_resultant_min      6141 non-null   float64
 3   acc_resultant_max      6141 non-null   float64
 4   acc_resultant_median   6141 non-null   float64
 5   acc_resultant_rms      6141 non-null   float64
 6   acc_horizontal_mean    6141 non-null   float64
 7   acc_horizontal_std     6141 non-null   float64
 8   acc_horizontal_min     6141 non-null   float64
 9   acc_horizontal_max     6141 non-null   float64
 10  acc_horizontal_median  6141 non-null   float64
 11  acc_horizontal_rms     6141 non-null   float64
 12  speed_mean             6141 non-null   float64
 13  speed_std              6141 non-null   float64
 14  speed_min              6141 non-null   float64
 15  spee

In [44]:
window_dataset.describe().T.head(10)

,count,mean,std,min,25%,50%,75%,max
acc_resultant_mean,6141.0,0.047292,0.016425,0.025106,0.037767,0.043955,0.051514,0.192325
acc_resultant_std,6141.0,0.022065,0.007668,0.008813,0.016708,0.020335,0.025136,0.068323
acc_resultant_min,6141.0,0.013424,0.009188,0.002449,0.008485,0.012124,0.015937,0.149255
acc_resultant_max,6141.0,0.101992,0.033322,0.048425,0.078147,0.096089,0.117533,0.252256
acc_resultant_median,6141.0,0.044174,0.016494,0.022136,0.035163,0.040792,0.048372,0.190751
acc_resultant_rms,6141.0,0.052199,0.017582,0.027482,0.041676,0.048389,0.057161,0.194533
acc_horizontal_mean,6141.0,0.041903,0.015994,0.021155,0.032904,0.038462,0.046111,0.187647
acc_horizontal_std,6141.0,0.022520,0.007764,0.010626,0.017192,0.020598,0.025595,0.067851
acc_horizontal_min,6141.0,0.008104,0.008269,0.000000,0.004243,0.006403,0.010000,0.139904
acc_horizontal_max,6141.0,0.098453,0.033721,0.042202,0.075186,0.091395,0.110766,0.251205


In [46]:
def load_accelerometer(acc_path):

    acc_columns = [

        "timestamp",
        "active",

        "acc_x",
        "acc_y",
        "acc_z",

        "acc_x_kf",
        "acc_y_kf",
        "acc_z_kf",

        "roll",
        "pitch",
        "yaw"

    ]

    acc_df = pd.read_csv(
        acc_path,
        sep=r"\s+",
        header=None,
        names=acc_columns
    )

    return acc_df